# Football AI Analysis on Colab
Detect players, ball, referees -- draw pitch keypoints, minimap, heatmap

## 1. Clone project

In [ ]:
!git clone https://github.com/henruysun2511/football_tracking.git
%cd football_tracking

## 2. Install dependencies

In [ ]:
!pip install -r requirements.txt
!pip install pyngrok

## 3. Download models

Do GitHub khong ho tro file >100MB, ban can tai models ve thu cong.

**Cach 1:** Upload `models/` tu may len Colab:
- O ben trai Colab, vao tab **Files** > **Upload to session storage**
- Upload ca thu muc `models/` (gom `player_detector.pt`, `pitch_keypoint_detector.pt`...)

**Cach 2:** Tai tu Google Drive (chay cell ben duoi sau khi mount)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy models tu Drive vao project
# Thay duong dan den thu muc models cua ban tren Drive
import os, shutil
SRC = '/content/drive/MyDrive/football_tracking_models/models'
DST = '/content/football_tracking/models'
if os.path.exists(SRC):
    if os.path.exists(DST):
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST)
    print('Models copied successfully')
    !ls -lh models/
else:
    print(f'NOT FOUND: {SRC}')
    print('Please upload models/ to Google Drive first.')

## 4. Upload input video

Dat file `.mp4` vao thu muc `input_videos/`:
- O tab **Files** ben trai, mo `football_tracking/input_videos/`
- Keo tha file video vao

In [ ]:
!ls -lh input_videos/

## 5. Run Streamlit

In [ ]:
import threading, time, sys, io

def run_streamlit():
    sys.stdout = io.StringIO()
    !streamlit run app.py --server.port 8501 --server.address 0.0.0.0

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(10)

In [ ]:
# Cai cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb > /dev/null 2>&1
!rm cloudflared.deb
print('cloudflared installed')

In [ ]:
import subprocess, time

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True)

time.sleep(5)
# Doc output de lay URL
output = proc.stderr.readline() if proc.stderr else ''
for _ in range(20):
    line = proc.stderr.readline() if proc.stderr else ''
    if 'trycloudflare.com' in line:
        url = line.split('https://')[1].split()[0]
        print(f'Open this URL: https://{url}')
        break
else:
    print('Could not get URL, check output above')

## 6. Get current URL (neu bi mat ket noi)

In [ ]:
import subprocess, time
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True)
time.sleep(5)
output = proc.stderr.readline() if proc.stderr else ''
for _ in range(20):
    line = proc.stderr.readline() if proc.stderr else ''
    if 'trycloudflare.com' in line:
        url = line.split('https://')[1].split()[0]
        print(f'Open this URL: https://{url}')
        break